# CRA-PI-GNN walkthrough (PyTorch Geometric backend)

This notebook walks through **CRA-PI-GNN** — *Continuous Relaxation Annealing* applied to a *Parameterised Independent* GNN — using QQA4CO's optional `qqa.pignn` backend (PyTorch Geometric port of the NeurIPS 2024 paper [*Controlling Continuous Relaxation for Combinatorial Optimization*](https://openreview.net/forum?id=ykACV1IhjD), reference DGL implementation: [Yuma-Ichikawa/CRA4CO](https://github.com/Yuma-Ichikawa/CRA4CO)).

We cover **all five problems** the PyG backend currently supports:

1. **Maximum Independent Set** (`qqa.MaximumIndependentSet`)
2. **MaxCut** (`qqa.MaxCut`)
3. **Max Clique** (`qqa.MaxClique`)
4. **Vertex Cover** (`qqa.VertexCover`)
5. **Graph Bisection** (`qqa.GraphBisection`)

For each problem we run **`train_cra_pi_gnn`** (the PyG re-implementation of CRA-PI-GNN) and the default **`qqa.anneal`** (parallel-replica annealing) on the same instance and seed for a head-to-head comparison.

> **Recommendation.** `qqa.anneal` is the *default* solver and is faster + more robust to hyperparameters across problem sizes. `qqa.pignn` is included so you can compare against the published CRA-PI-GNN paper from the same codebase, on Blackwell-class GPUs where DGL prebuilt wheels are not yet available. See the README's "Empirical comparison" table.

**Hyperparameters note.** The defaults in `train_cra_pi_gnn` (`init_reg_param=-20`, `annealing_rate=1e-3`, `learning_rate=1e-4`, `num_epochs=1e5`) are tuned for the paper's `N >= 1000` regime. For the small instances used in this notebook we use the medium-graph regime `init_reg_param=-2.0, annealing_rate=5e-4, learning_rate=1e-3` documented in `qqa/pignn/trainer.py`'s docstring.

## 0. Install (run once)

The PyG backend is opt-in. Install it with the `pignn` extra:

```bash
pip install "qqa[pignn]"
```

If you cloned the repo and use `uv`:

```bash
uv sync --extra pignn
```

In [1]:
import time

import networkx as nx
import torch

import qqa
from qqa.pignn import train_cra_pi_gnn

SEED = 0
qqa.fix_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"qqa version : {qqa.__version__}")
print(f"torch       : {torch.__version__}  (CUDA available: {torch.cuda.is_available()})")
print(f"device      : {device}")

# Hyperparameters used throughout this notebook. They are the medium-graph
# regime (N ~ 100-300) recommended in qqa/pignn/trainer.py — the paper
# defaults under-converge on small graphs.
PIGNN_HP = dict(
    learning_rate=1e-3,
    init_reg_param=-2.0,
    annealing_rate=5e-4,
    num_epochs=4000,
    check_interval=10_000,  # silences per-step prints
    verbose=False,
    seed=SEED,
    device=device,
)

# qqa.anneal hyperparameters: rely on the library defaults
# (see ``qqa.annealing.anneal``) and just shorten ``num_epochs`` so the
# notebook runs quickly on CPU. Letting ``min_bg`` / ``max_bg`` default
# to ``None`` allows the schedule to pick problem-appropriate values.
QQA_HP = dict(
    sol_size=100,
    num_epochs=4000,
    device=device,
    verbose=False,
)


def _bench(fn, *args, **kwargs):
    """Tiny helper: returns (result, elapsed_seconds)."""
    t0 = time.perf_counter()
    res = fn(*args, **kwargs)
    return res, time.perf_counter() - t0

qqa version : 0.3.0
torch       : 2.11.0+cu130  (CUDA available: False)
device      : cpu


## 1. Maximum Independent Set (MIS)

Find the largest vertex set with no two adjacent vertices.

QUBO: `H = -|S| + penalty * (#violated edges)` where `S = {i : x_i = 1}`.

We use a 200-node 3-regular random graph (Caro-Wei lower bound: `|IS| / N ~ 0.36`, so the optimum is around `|IS| ~ 72`).

In [2]:
qqa.fix_seed(SEED)
g_mis = nx.random_regular_graph(d=3, n=200, seed=SEED)
problem_mis = qqa.MaximumIndependentSet(g_mis, penalty=2, device=device)

print(f"[MIS] N={problem_mis.num_nodes}  |E|={g_mis.number_of_edges()}")

res_pignn, t_pignn = _bench(train_cra_pi_gnn, problem_mis, **PIGNN_HP)
res_qqa, t_qqa = _bench(qqa.anneal, problem_mis, **QQA_HP)

print(
    f"  CRA-PI-GNN : |IS| = {int(res_pignn.score['value']):>3d}  "
    f"({'feasible' if res_pignn.score['feasible'] else 'INFEASIBLE'})  "
    f"runtime = {t_pignn:5.1f}s"
)
print(
    f"  qqa.anneal : |IS| = {int(res_qqa.score['value']):>3d}  "
    f"({'feasible' if res_qqa.score['feasible'] else 'INFEASIBLE'})  "
    f"runtime = {t_qqa:5.1f}s"
)

[MIS] N=200  |E|=300


/lustre/home/frj/yuma-ichikawa/research/papers/QQA4CO/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  CRA-PI-GNN : |IS| =  79  (feasible)  runtime =  71.6s
  qqa.anneal : |IS| =  86  (INFEASIBLE)  runtime =   8.1s


## 2. MaxCut

Partition the vertices into two sets to maximise the number of edges crossing the cut.

QUBO: `H = -sum_{(u,v) in E} (x_u + x_v - 2 x_u x_v)` (`= -cut_size`).

In [3]:
qqa.fix_seed(SEED)
g_cut = nx.random_regular_graph(d=3, n=200, seed=SEED)
problem_cut = qqa.MaxCut(g_cut, device=device)

print(f"[MaxCut] N={problem_cut.num_nodes}  |E|={g_cut.number_of_edges()}")

res_pignn, t_pignn = _bench(train_cra_pi_gnn, problem_cut, **PIGNN_HP)
res_qqa, t_qqa = _bench(qqa.anneal, problem_cut, **QQA_HP)

print(
    f"  CRA-PI-GNN : cut = {int(res_pignn.score['value']):>3d}  "
    f"runtime = {t_pignn:5.1f}s"
)
print(
    f"  qqa.anneal : cut = {int(res_qqa.score['value']):>3d}  "
    f"runtime = {t_qqa:5.1f}s"
)

[MaxCut] N=200  |E|=300


  CRA-PI-GNN : cut = 270  runtime =  62.8s
  qqa.anneal : cut = 273  runtime =   8.7s


## 3. Max Clique

Find the largest fully-connected subgraph (clique). Equivalent to MIS on the *complement* graph.

We use a 100-node Erdős–Rényi `G(N, p)` with `p = 0.6` (denser than the MIS instance so non-trivial cliques exist).

In [4]:
qqa.fix_seed(SEED)
# Sparser graph + larger penalty so neither solver collapses to the trivial
# empty-clique solution (which is feasible but useless).
g_clq = nx.erdos_renyi_graph(n=100, p=0.5, seed=SEED)
problem_clq = qqa.MaxClique(g_clq, penalty=4, device=device)

print(f"[MaxClique] N={problem_clq.num_nodes}  |E|={g_clq.number_of_edges()}")

res_pignn, t_pignn = _bench(train_cra_pi_gnn, problem_clq, **PIGNN_HP)
res_qqa, t_qqa = _bench(qqa.anneal, problem_clq, **QQA_HP)

print(
    f"  CRA-PI-GNN : |clique| = {int(res_pignn.score['value']):>2d}  "
    f"({'feasible' if res_pignn.score['feasible'] else 'INFEASIBLE'})  "
    f"runtime = {t_pignn:5.1f}s"
)
print(
    f"  qqa.anneal : |clique| = {int(res_qqa.score['value']):>2d}  "
    f"({'feasible' if res_qqa.score['feasible'] else 'INFEASIBLE'})  "
    f"runtime = {t_qqa:5.1f}s"
)

[MaxClique] N=100  |E|=2444


  CRA-PI-GNN : |clique| =  1  (feasible)  runtime =  14.2s
  qqa.anneal : |clique| =  5  (feasible)  runtime =   6.6s


## 4. Vertex Cover

Find the smallest vertex set that touches every edge.

QUBO: `H = sum_i x_i + penalty * sum_{(u,v) in E} (1 - x_u)(1 - x_v)`.

Note that `VertexCover` exposes its graph on `problem.graph` (not `problem.nx_graph`); `qqa.pignn.graph.extract_nx_graph` searches both attribute names automatically.

In [5]:
qqa.fix_seed(SEED)
g_vc = nx.random_regular_graph(d=3, n=100, seed=SEED)
problem_vc = qqa.VertexCover(g_vc, penalty=4.0, device=device)

print(f"[VertexCover] N={problem_vc.num_nodes}  |E|={g_vc.number_of_edges()}")

res_pignn, t_pignn = _bench(train_cra_pi_gnn, problem_vc, **PIGNN_HP)
res_qqa, t_qqa = _bench(qqa.anneal, problem_vc, **QQA_HP)

print(
    f"  CRA-PI-GNN : cover size = {int(res_pignn.score['value']):>3d}  "
    f"({'feasible' if res_pignn.score['feasible'] else 'INFEASIBLE'})  "
    f"runtime = {t_pignn:5.1f}s"
)
print(
    f"  qqa.anneal : cover size = {int(res_qqa.score['value']):>3d}  "
    f"({'feasible' if res_qqa.score['feasible'] else 'INFEASIBLE'})  "
    f"runtime = {t_qqa:5.1f}s"
)

[VertexCover] N=100  |E|=150


  CRA-PI-GNN : cover size =  58  (feasible)  runtime =   8.6s
  qqa.anneal : cover size =  56  (feasible)  runtime =   7.5s


## 5. Graph Bisection

Partition vertices into two equal-size sets minimising the cut.

QUBO: `H = sum_{(u,v) in E} (x_u - x_v)^2 + balance_penalty * (sum_i x_i - N/2)^2`.

Like `VertexCover`, this problem stores its graph on `problem.graph`.

In [6]:
qqa.fix_seed(SEED)
g_gb = nx.random_regular_graph(d=3, n=100, seed=SEED)
problem_gb = qqa.GraphBisection(g_gb, balance_penalty=1.0, device=device)

print(f"[GraphBisection] N={problem_gb.num_nodes}  |E|={g_gb.number_of_edges()}")

res_pignn, t_pignn = _bench(train_cra_pi_gnn, problem_gb, **PIGNN_HP)
res_qqa, t_qqa = _bench(qqa.anneal, problem_gb, **QQA_HP)

print(
    f"  CRA-PI-GNN : cut = {int(res_pignn.score['value']):>3d}  "
    f"({'feasible' if res_pignn.score['feasible'] else 'INFEASIBLE'})  "
    f"runtime = {t_pignn:5.1f}s"
)
print(
    f"  qqa.anneal : cut = {int(res_qqa.score['value']):>3d}  "
    f"({'feasible' if res_qqa.score['feasible'] else 'INFEASIBLE'})  "
    f"runtime = {t_qqa:5.1f}s"
)

[GraphBisection] N=100  |E|=150


  CRA-PI-GNN : cut =  44  (feasible)  runtime =  11.2s
  qqa.anneal : cut =  17  (INFEASIBLE)  runtime =   8.5s


## Wrap-up

You just ran **CRA-PI-GNN** (PyG port of the NeurIPS 2024 reference implementation) and **QQA** on five QUBO-on-a-graph problems through a single import (`import qqa`).

### About the numbers above

This notebook is a **usage walkthrough**, not a tuned benchmark. To keep the runtime under five minutes on CPU we shrank both solvers' epoch counts and held all other hyperparameters fixed across five very different problems (sparse vs. dense, hard vs. easy constraints). As a result, you may see:

- **`qqa.anneal` returning `INFEASIBLE`** on MIS / GraphBisection at this size — the parallel-replica annealer needs more epochs (or a stronger `max_bg`) to drive constraint violations to zero on these problem instances. Bumping `num_epochs` to 10k (the library default) typically fixes this.
- **CRA-PI-GNN collapsing to a near-trivial solution** on Max Clique — small dense graphs are CRA-PI-GNN's weakest regime. The paper's headline numbers use `N >= 1000`, `init_reg_param=-20`, `num_epochs=1e5`; on tiny instances per-problem retuning of `init_reg_param` and `annealing_rate` is required.

For honest, *tuned* head-to-head numbers see `scripts/bench_qqa_vs_pignn.py` and the "Empirical comparison" table in the README.

### Where to go next

- **Larger benchmarks**: `scripts/bench_qqa_vs_pignn.py` runs the same head-to-head on `N = {100, 300, 500}` and prints a markdown table you can paste into a paper.
- **Blackwell / B200 GPU sanity check**: `scripts/sanity_pignn_gpu.sbatch` (Slurm) verifies `sm_100` is in `torch.cuda.get_arch_list()` and runs the demo on the GPU.
- **CLI**: `qqa solve --problem mis --backend pignn --learning-rate 1e-3 --pignn-init-reg-param -2 --pignn-annealing-rate 5e-4 --epochs 5000`.
- **Reference DGL implementation**: <https://github.com/Yuma-Ichikawa/CRA4CO> (NeurIPS 2024).
- **QQA paper**: Ichikawa & Arai, *"Optimization by Parallel Quasi-Quantum Annealing with Gradient-Based Sampling"*, ICLR 2025 ([OpenReview](https://openreview.net/forum?id=9EfBeXaXf0), [arXiv:2409.02135](https://arxiv.org/abs/2409.02135)).
- **CPRA paper (CRA / CPRA backends)**: Ichikawa & Iwashita, *"Continuous Parallel Relaxation for Finding Diverse Solutions in Combinatorial Optimization Problems"*, TMLR 2025 ([OpenReview](https://openreview.net/forum?id=ix33zd5zCw)).